In [ ]:
import pandas as pd
from tqdm import tqdm
import math

In [ ]:
inds = {}

trios_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = int(ind_idx.split('_')[0])
    inds[ind] = True

In [ ]:
sibs = {}

sibs_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info.txt', sep = '\t')

for _, row in sibs_dnms_df.iterrows():
    
    sib_idx = row['IID']
    sibs[sib_idx] = True

sibs_df = pd.read_csv('/mnt/project/DNA repair genes/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    b = int(i/100)
    p = i%100
            
    sib1_idx = f'{sib1}_b{b}_p{p}'
    sib2_idx = f'{sib2}_b{b}_p{p}'
        
    if sib1 in inds and sib2 in inds:
        continue
        
    if sib1_idx not in sibs and sib2_idx not in sibs:
        continue

    elif sib1_idx not in sibs:
        sibs[sib1_idx] = True
    elif sib2_idx not in sibs:
        sibs[sib2_idx] = True
    
    inds[(sib1, sib2)] = True

In [ ]:
len(inds)

In [ ]:
genes_ukb = {}

genes_ukb_df = pd.read_csv('/mnt/project/DNA repair genes/repair_genes_map.txt', sep = ' ', header = None)

for _, row in genes_ukb_df.iterrows():
    if row[3] == 'X':
        continue
    genes_ukb[row[0]] = True

# Read LoF and missense mutation data

In [ ]:
lofs_map_df = pd.read_csv('/mnt/project/DNA repair genes/repair_genes_lofs_map.txt', sep = ' ', header = None)
lofs_map = dict(zip(lofs_map_df[0], lofs_map_df[1]))
lofs_map_df.head()

In [ ]:
miss_map_df = pd.read_csv('/mnt/project/DNA repair genes/repair_genes_miss_map.txt', sep = ' ', header = None)
miss_map = dict(zip(miss_map_df[0], miss_map_df[1]))
miss_map_df.head()

In [ ]:
max_maf = 0

lofs_macs = {}

for chrom in tqdm(range(1, 23)):
    
    if chrom == 21:
        continue

    lofs_macs_df = pd.read_csv(f'/mnt/project/DNA repair genes/MAC/repair_genes_lofs_chr{chrom}.acount', sep = '\t')

    for _, row in lofs_macs_df.iterrows():

        lof = row['ID'][7:]
        mac = int(row['ALT_CTS'])
        obs = int(row['OBS_CT'])
        lofs_macs[lof] = (mac, obs)
        if mac/obs > max_maf:
            max_maf = mac/obs

print(max_maf)

In [ ]:
max_maf = 0

miss_macs = {}

for chrom in tqdm(range(1, 23)):
    
    if chrom == 21:
        continue

    miss_macs_df = pd.read_csv(f'/mnt/project/DNA repair genes/MAC/repair_genes_miss_chr{chrom}.acount', sep = '\t')

    for _, row in miss_macs_df.iterrows():

        mis = row['ID'][7:]
        mac = int(row['ALT_CTS'])
        obs = int(row['OBS_CT'])
        miss_macs[mis] = (mac, obs)
        if mac/obs > max_maf:
            max_maf = mac/obs

print(max_maf)

In [ ]:
gene_lofs = {gene: [] for gene in genes_ukb}

for lof in lofs_map:
    
    if ((lofs_macs[lof][0]/lofs_macs[lof][1]) > 0.01):
        continue
    
    gene_lofs[lofs_map[lof]].append(lof)

In [ ]:
gene_miss = {gene: [] for gene in genes_ukb}

for mis in miss_map:
    gene_miss[miss_map[mis]].append(mis)

In [ ]:
inds_lofs = {}

for chrom in tqdm(range(1, 23)):
    
    if chrom == 21:
        continue
    
    with open(f'/mnt/project/DNA repair genes/counts/lofs_chr{chrom}.tsv') as f:
        
        for row in f:
            
            row = row.strip()
            row = row.split()
            
            ind = int(row[0].split('_')[1])
            lofs = row[1].split('|')[1:] if len(row) > 1 else []
            
            if ind not in inds_lofs:
                inds_lofs[ind] = {}
                
            for lof in lofs:
                
                chrom = int(lof.split(':')[0])
                pos = int(lof.split(':')[1])
                ref = lof.split(':')[2]
                alt = lof.split(':')[3]
                
                if ((lofs_macs[f'chr{chrom}:{pos}:{ref}:{alt}'][0]/
                     lofs_macs[f'chr{chrom}:{pos}:{ref}:{alt}'][1]) > 0.01):
                    continue

                if f'chr{chrom}:{pos}:{ref}:{alt}' not in lofs_map:
                    continue
                
                gt = (float(lof.split(':')[4][0]) +
                      float(lof.split(':')[4][2]))
                
                gene = lofs_map[f'chr{chrom}:{pos}:{ref}:{alt}']
                inds_lofs[ind][f'chr{chrom}:{pos}:{ref}:{alt}'] = (gt, gene)

In [ ]:
inds_miss = {}

for chrom in tqdm(range(1, 23)):
    
    if chrom == 21:
        continue

    with open(f'/mnt/project/DNA repair genes/counts/miss_chr{chrom}.tsv') as f:
        
        for row in f:
            
            row = row.strip()
            row = row.split()
            
            ind = int(row[0].split('_')[1])
            miss = row[1].split('|')[1:] if len(row) > 1 else []
            
            if ind not in inds_miss:
                inds_miss[ind] = {}
                
            for mis in miss:

                chrom = int(mis.split(':')[0])
                pos = int(mis.split(':')[1])
                ref = mis.split(':')[2]
                alt = mis.split(':')[3]
                
                if f'chr{chrom}:{pos}:{ref}:{alt}' not in miss_map:
                    continue
                
                gt = (float(mis.split(':')[4][0]) +
                      float(mis.split(':')[4][2]))
                
                gene = miss_map[f'chr{chrom}:{pos}:{ref}:{alt}']
                inds_miss[ind][f'chr{chrom}:{pos}:{ref}:{alt}'] = (gt, gene)

# Trios

In [ ]:
trios_df = pd.read_csv('/mnt/project/DNA repair genes/trios_pairs.txt', sep = ' ', header = None)

In [ ]:
inds_lofs_genes = {}

for _, row in tqdm(trios_df.iterrows()):
    
    ind = int(row[0])
    father = int(row[1])
    mother = int(row[2])

    if ind not in inds:
        continue
    
    if father not in inds_lofs or mother not in inds_lofs:
        continue
        
    inds_lofs_genes[ind] = {gene: 0.0 for gene in genes_ukb}
    
    for lof in inds_lofs[father]:
        gt = inds_lofs[father][lof][0]
        gene = inds_lofs[father][lof][1]
        if gene not in genes_ukb:
            continue
        inds_lofs_genes[ind][gene] += gt
        
    for lof in inds_lofs[mother]:
        gt = inds_lofs[mother][lof][0]
        gene = inds_lofs[mother][lof][1]
        if gene not in genes_ukb:
            continue
        inds_lofs_genes[ind][gene] += gt

In [ ]:
inds_miss_genes = {}

for _, row in tqdm(trios_df.iterrows()):
    
    ind = int(row[0])
    father = int(row[1])
    mother = int(row[2])

    if ind not in inds:
        continue
    
    if father not in inds_miss or mother not in inds_miss:
        continue
    
    inds_miss_genes[ind] = {gene: 0.0 for gene in genes_ukb}
    
    for mis in inds_miss[father]:
        gt = inds_miss[father][mis][0]
        gene = inds_miss[father][mis][1]
        if gene not in genes_ukb:
            continue
        inds_miss_genes[ind][gene] += gt
        
    for mis in inds_miss[mother]:
        gt = inds_miss[mother][mis][0]
        gene = inds_miss[mother][mis][1]
        if gene not in genes_ukb:
            continue
        inds_miss_genes[ind][gene] += gt

In [ ]:
len(inds_lofs_genes)

In [ ]:
len(inds_miss_genes)

# Sibs

In [ ]:
def read_ibd(n):
    
    ibd_temp = {}

    with open(f'/mnt/project/DNM/snipar/sibs_ibd{n}.bed') as f:

        for row in f:

            row = row.strip()

            if '>start' in row:
                row = row.split(':')[1]
                sib1 = int(row.split('-')[0])
                sib2 = int(row.split('-')[1])
                ibd_temp[(sib1, sib2)] = {chrom: [] for chrom in range(1, 23)}

            elif '>end' not in row:
                row = row.split(' ')
                chrom = int(row[0])
                start = int(row[1])
                end = int(row[2])
                ibd_temp[(sib1, sib2)][chrom].append((start, end))
                
    return ibd_temp

In [ ]:
ibd = {}
for n in range(3):
    ibd[n] = read_ibd(n)

In [ ]:
def get_ibd(sib1, sib2, chrom, pos):
    
    for n in range(3):
        for start, end in ibd[n][(sib1, sib2)][chrom]:
            if pos >= start and pos <= end:
                return n
            
    return None

In [ ]:
def missing_inconsistent_ibd(sib1_gt, sib2_gt, q):

    if sib1_gt == 0 and sib2_gt == 0:
        return 2*q/(2-q)

    elif (sib1_gt == 0 and sib2_gt == 1) or (sib1_gt == 1 and sib2_gt == 0):
        return 2/(2-q)

    elif (sib1_gt == 0 and sib2_gt == 2) or (sib1_gt == 2 and sib2_gt == 0):
        return 2.0

    elif sib1_gt == 1 and sib2_gt == 1:
        return 2+(2*q-1)/(1+q*(1-q))

    elif (sib1_gt == 1 and sib2_gt == 2) or (sib1_gt == 2 and sib2_gt == 1):
        return 2*(1+2*q)/(1+q)

    elif sib1_gt == 2 and sib2_gt == 2:
        return 2*(1+3*q)/(1+q)

In [ ]:
sibs_lofs = {}

for _, row in tqdm(sibs_df.iterrows()):
    
    sib1 = int(row[0])
    sib2 = int(row[1])

    if (sib1, sib2) not in inds:
        continue
    
    if sib1 in inds_lofs_genes and sib2 in inds_lofs_genes:
        continue
    
    if sib1 not in inds_lofs or sib2 not in inds_lofs:
        continue
        
    sibs_lofs[(sib1, sib2)] = {}
    
    for lof in lofs_map:

        q = lofs_macs[lof][0]/lofs_macs[lof][1]
        p = 1-q

        if q > 0.01:
            continue

        sib1_gt = (inds_lofs[sib1][lof][0] 
                   if lof in inds_lofs[sib1] else 0.0)
        sib2_gt = (inds_lofs[sib2][lof][0] 
                   if lof in inds_lofs[sib2] else 0.0)
        
        chrom = int(lof.split(':')[0][3:])
        pos = int(lof.split(':')[1])
        n = get_ibd(sib1, sib2, chrom, pos)
    
        if n == None:
            sibs_lofs[(sib1, sib2)][lof] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
            continue
            
        if n == 0:
            sibs_lofs[(sib1, sib2)][lof] = ((sib1_gt + sib2_gt), 'IBD0')
            
        elif n == 1:
            
            if abs(sib1_gt-sib2_gt) == 2:
                sibs_lofs[(sib1, sib2)][lof] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
                continue
            
            if sib1_gt == 0 and sib2_gt == 0:
                sibs_lofs[(sib1, sib2)][lof] = (q, 'IBD1')
                
            elif ((sib1_gt == 0 and sib2_gt == 1) or
                  (sib1_gt == 1 and sib2_gt == 0)):
                sibs_lofs[(sib1, sib2)][lof] = ((1 + q), 'IBD1')
                
            elif sib1_gt == 1 and sib2_gt == 1:
                sibs_lofs[(sib1, sib2)][lof] = ((1 + q*2), 'IBD1')
                
            elif ((sib1_gt == 1 and sib2_gt == 2) or
                  (sib1_gt == 2 and sib2_gt == 1)):
                sibs_lofs[(sib1, sib2)][lof] = ((2 + q), 'IBD1')
                
            elif sib1_gt == 2 and sib2_gt == 2:
                sibs_lofs[(sib1, sib2)][lof] = ((3 + q), 'IBD1')
            
        elif n == 2:
            
            if sib1_gt != sib2_gt:
                sibs_lofs[(sib1, sib2)][lof] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
                continue
                
            sibs_lofs[(sib1, sib2)][lof] = ((sib1_gt + q*2), 'IBD2')

In [ ]:
sibs_miss = {}

for _, row in tqdm(sibs_df.iterrows()):
    
    sib1 = int(row[0])
    sib2 = int(row[1])

    if (sib1, sib2) not in inds:
        continue
    
    if sib1 in inds_miss_genes and sib2 in inds_miss_genes:
        continue
    
    if sib1 not in inds_miss or sib2 not in inds_miss:
        continue
        
    sibs_miss[(sib1, sib2)] = {}
    
    for mis in miss_map:

        q = miss_macs[mis][0]/miss_macs[mis][1]
        p = 1-q
        
        sib1_gt = (inds_miss[sib1][mis][0] 
                   if mis in inds_miss[sib1] else 0.0)
        sib2_gt = (inds_miss[sib2][mis][0] 
                   if mis in inds_miss[sib2] else 0.0)
        
        chrom = int(mis.split(':')[0][3:])
        pos = int(mis.split(':')[1])
        n = get_ibd(sib1, sib2, chrom, pos)
    
        if n == None:
            sibs_miss[(sib1, sib2)][mis] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
            continue
            
        if n == 0:
            sibs_miss[(sib1, sib2)][mis] = ((sib1_gt + sib2_gt), 'IBD0')
            
        elif n == 1:
            
            if abs(sib1_gt-sib2_gt) == 2:
                sibs_miss[(sib1, sib2)][mis] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
                continue
            
            if sib1_gt == 0 and sib2_gt == 0:
                sibs_miss[(sib1, sib2)][mis] = (q, 'IBD1')
                
            elif ((sib1_gt == 0 and sib2_gt == 1) or
                  (sib1_gt == 1 and sib2_gt == 0)):
                sibs_miss[(sib1, sib2)][mis] = ((1 + q), 'IBD1')
                
            elif sib1_gt == 1 and sib2_gt == 1:
                sibs_miss[(sib1, sib2)][mis] = ((1 + q*2), 'IBD1')
                
            elif ((sib1_gt == 1 and sib2_gt == 2) or
                  (sib1_gt == 2 and sib2_gt == 1)):
                sibs_miss[(sib1, sib2)][mis] = ((2 + q), 'IBD1')
                
            elif sib1_gt == 2 and sib2_gt == 2:
                sibs_miss[(sib1, sib2)][mis] = ((3 + q), 'IBD1')
            
        elif n == 2:
            
            if sib1_gt != sib2_gt:
                sibs_miss[(sib1, sib2)][mis] = (missing_inconsistent_ibd(sib1_gt, sib2_gt, q), 'NA')
                continue
                
            sibs_miss[(sib1, sib2)][mis] = ((sib1_gt + q*2), 'IBD2')

In [ ]:
len(sibs_lofs)

In [ ]:
len(sibs_miss)

# Families

In [ ]:
fams = {}
ind_fam = {}

fams_df = pd.read_csv('ukb_families.csv')

for _, row in fams_df.iterrows():
    i = row['ID']
    fams[i] = [int(ind) for ind in row['INDS'].split(',')]
    for ind in fams[i]:
        ind_fam[ind] = i

In [ ]:
fam_inds = {i: [] for i in fams}

for ind in inds_lofs_genes:
    if ind in ind_fam:
        i = ind_fam[ind]
        fam_inds[i].append(ind)

for sib1, sib2 in sibs_lofs:
    if sib1 in ind_fam and sib2 in ind_fam:
        i = ind_fam[sib1]
        fam_inds[i].append((sib1, sib2))

In [ ]:
len(fam_inds)

In [ ]:
order = {'IBD0': 0, 'IBD1': 1, 'IBD2': 2, 'NA': 3}

def custom_sort(x):
    gt, ibd = x
    if ibd == 'IBD0':
        return (0, -gt, 0, 0)
    if ibd == 'NA':
        return (2, -gt, 0, 0)
    return (1, -math.floor(gt), order[ibd], -gt)

fams_lofs_genes = {}
fams_miss_genes = {}

for _, i in tqdm(enumerate(fam_inds)):

    fams_lofs_genes[i] = {}
    fams_miss_genes[i] = {}
    
    if isinstance(fam_inds[i][0], int):
        for gene in genes_ukb:
            fams_lofs_genes[i][gene] = inds_lofs_genes[fam_inds[i][0]][gene]
            fams_miss_genes[i][gene] = inds_miss_genes[fam_inds[i][0]][gene]

    elif isinstance(fam_inds[i][0], tuple):
        for gene in genes_ukb:
            fams_lofs_genes[i][gene] = sum(min((sibs_lofs[pair][lof] for pair in fam_inds[i]),
                                               key = custom_sort, default = (0.0, 'NA'))[0] for lof in gene_lofs[gene])
            fams_miss_genes[i][gene] = sum(min((sibs_miss[pair][mis] for pair in fam_inds[i]),
                                               key = custom_sort, default = (0.0, 'NA'))[0] for mis in gene_miss[gene])

In [ ]:
len(fams_lofs_genes)

In [ ]:
len(fams_miss_genes)

# Record genotypes

In [ ]:
rows = {}

for i in fams_lofs_genes:
    fam = f'{fams[i][0]}'
    for ind in fams[i][1:]:
        fam += f'_{ind}'
    rows[fam] = fams_lofs_genes[i]

df = pd.DataFrame.from_dict(rows, orient = 'index')
df.to_csv('lofs_genes_ukb.csv', index_label = 'FAM')

In [ ]:
rows = {}

for i in fams_miss_genes:
    fam = f'{fams[i][0]}'
    for ind in fams[i][1:]:
        fam += f'_{ind}'
    rows[fam] = fams_miss_genes[i]

df = pd.DataFrame.from_dict(rows, orient = 'index')
df.to_csv('miss_genes_ukb.csv', index_label = 'FAM')